# Airbnb Price Prediction

## Objectif
Prédire le **prix de nuit** de logements Airbnb à partir de leurs caractéristiques.

## Données
- **Train** : 22 235 annonces avec le prix connu
- **Test** : 51 881 annonces dont on doit prédire le prix
- Variables disponibles : type de logement, ville, quartier, équipements, infos hôte, avis...

## Étapes du projet
1. **Explorer les données** — comprendre ce qui influence le prix
2. **Préparer les features** — transformer les colonnes brutes en nombres utilisables
3. **Encoder les catégories** — convertir les textes en valeurs numériques
4. **Entraîner un modèle** — LightGBM avec validation croisée
5. **Analyser les résultats** — vérifier que le modèle est cohérent

> La variable cible est `log_price` (le logarithme du prix).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
import re
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

try:
    import lightgbm as lgb
    USE_LGBM = True
    print(f"LightGBM {lgb.__version__} disponible")
except ImportError:
    USE_LGBM = False
    print("LightGBM non disponible — fallback GradientBoostingRegressor")

plt.rcParams['figure.figsize'] = (12, 5)
pd.set_option('display.max_columns', 50)
SEED = 42
np.random.seed(SEED)

## 1. Chargement des données

In [ ]:
train = pd.read_csv('airbnb_train.csv')
test  = pd.read_csv('airbnb_test.csv')

# La 1ère colonne du test est sans nom (c'est l'id)
test.rename(columns={test.columns[0]: 'id'}, inplace=True)

print(f"Train : {train.shape[0]:,} lignes x {train.shape[1]} colonnes")
print(f"Test  : {test.shape[0]:,} lignes x {test.shape[1]} colonnes")
train.head(3)

## 2. Exploration des données (EDA)

In [ ]:
# Valeurs manquantes
def missing_report(df, name):
    m = (df.isnull() | (df == '')).sum()
    m = m[m > 0].sort_values(ascending=False)
    pct = (m / len(df) * 100).round(1)
    print(f"\n--- {name} ---")
    print(pd.DataFrame({'manquants': m, '%': pct}).to_string())

missing_report(train, 'Train')

### Observations : valeurs manquantes

Certaines colonnes ont beaucoup de valeurs manquantes :
- `host_response_rate` : environ 40% des hôtes ne renseignent pas ce champ
- `review_scores_rating`, `first_review`, `last_review` : vides pour les nouvelles annonces sans avis

**Ce qu'on va faire** : créer une colonne supplémentaire pour indiquer si la date est absente (c'est une info utile), et remplacer les valeurs numériques manquantes par la médiane.

In [ ]:
# Distribution de la variable cible
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(train['log_price'], bins=60, color='steelblue', edgecolor='white', lw=0.5)
axes[0].axvline(train['log_price'].mean(), color='red', ls='--',
                label=f"Moyenne : {train['log_price'].mean():.2f}")
axes[0].set_title('Distribution de log_price'); axes[0].set_xlabel('log_price')
axes[0].legend()

axes[1].hist(np.exp(train['log_price']), bins=100, color='coral', edgecolor='white', lw=0.5)
axes[1].set_xlim(0, 800); axes[1].set_title('Distribution du prix ($)')
axes[1].set_xlabel('Prix ($)')

plt.suptitle('Variable cible', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()
print(f"min={train['log_price'].min():.2f}  max={train['log_price'].max():.2f}  "
      f"mean={train['log_price'].mean():.2f}  std={train['log_price'].std():.2f}")

In [ ]:
# Prix moyen par catégorie
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, col in zip(axes.flatten(), ['room_type','property_type','city','cancellation_policy']):
    means = train.groupby(col)['log_price'].mean().sort_values(ascending=False)
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(means)))
    bars = ax.bar(range(len(means)), means.values, color=colors, edgecolor='white')
    ax.set_xticks(range(len(means)))
    ax.set_xticklabels(means.index, rotation=45, ha='right', fontsize=9)
    ax.set_title(f'log_price moyen par {col}', fontsize=11)
    for bar, val in zip(bars, means.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.02,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout(); plt.show()

### Observations : prix par catégorie

- Louer un logement entier coûte environ 2x plus cher qu'une chambre privée
- Bateaux, châteaux et villas ont les prix les plus élevés
- San Francisco et New York sont les villes les plus chères
- Les hôtes avec des règles d'annulation strictes proposent souvent des logements haut de gamme

Ces variables ont donc un vrai impact sur le prix et seront utiles au modèle.

In [ ]:
# Corrélations numériques
num_cols = ['accommodates','bathrooms','bedrooms','beds','number_of_reviews','review_scores_rating']
tmp = train[num_cols + ['log_price']].copy()
for c in num_cols:
    tmp[c] = pd.to_numeric(tmp[c], errors='coerce')
corr = tmp.corr()['log_price'].drop('log_price').sort_values(key=abs, ascending=False)
print("Corrélations avec log_price:")
print(corr.round(3))

### Observations : corrélations numériques

- Plus le logement est grand (voyageurs, chambres, salles de bain), plus il est cher
- Les logements avec beaucoup d'avis sont souvent moins chers (les logements abordables tournent davantage)
- La note des avis a peu d'effet direct sur le prix

La taille du logement est donc le principal facteur numérique.

## 2.2 Visualisations Avancées

Pour approfondir la compréhension des données :
- **Carte géographique** des annonces colorée par prix (rouge = cher, vert = pas cher)
- **Heatmap de corrélation** entre toutes les variables numériques
- **Boxplot** comparant la distribution des prix par ville
- **Impact des amenities** : quels équipements font monter ou baisser le prix ?
- **Violin plot** par type de logement + courbe prix / capacité d'accueil

In [ ]:
# ── Carte géographique des prix (rouge = cher, vert = pas cher) ───────────────
prices = np.exp(train['log_price'])
p5, p95 = np.percentile(prices, 5), np.percentile(prices, 95)

fig, ax = plt.subplots(figsize=(15, 9))
sc = ax.scatter(
    train['longitude'], train['latitude'],
    c=prices.clip(p5, p95),
    cmap='RdYlGn_r',
    s=6, alpha=0.45,
    vmin=p5, vmax=p95
)
cbar = plt.colorbar(sc, ax=ax, label='Prix par nuit ($)', fraction=0.03)
cbar.ax.tick_params(labelsize=9)

for city, grp in train.groupby('city'):
    cx, cy = grp['longitude'].median(), grp['latitude'].median()
    ax.annotate(city, (cx, cy), fontsize=10, fontweight='bold', color='white',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='#333333', alpha=0.8))

ax.set_xlabel('Longitude', fontsize=11)
ax.set_ylabel('Latitude', fontsize=11)
ax.set_title('Carte des annonces Airbnb — rouge = cher, vert = pas cher', fontsize=14, pad=12)
ax.set_facecolor('#e8e8e8')
fig.patch.set_facecolor('#f5f5f5')
plt.tight_layout()
plt.show()
print(f"Prix médian : \${np.median(prices):.0f}/nuit  |  Prix max : \${prices.max():.0f}/nuit")

In [ ]:
# ── Heatmap de corrélation ────────────────────────────────────────────────────
num_cols_viz = ['log_price', 'accommodates', 'bathrooms', 'bedrooms', 'beds',
                'number_of_reviews', 'review_scores_rating']
labels_fr = {'log_price': 'Prix (log)', 'accommodates': 'Voyageurs',
             'bathrooms': 'Salles de bain', 'bedrooms': 'Chambres', 'beds': 'Lits',
             'number_of_reviews': 'Nb avis', 'review_scores_rating': 'Note'}

tmp_corr = train[num_cols_viz].copy()
for c in num_cols_viz:
    tmp_corr[c] = pd.to_numeric(tmp_corr[c], errors='coerce')
tmp_corr.rename(columns=labels_fr, inplace=True)

corr = tmp_corr.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.8, linecolor='white',
            annot_kws={'size': 10}, ax=ax, vmin=-1, vmax=1,
            cbar_kws={'shrink': 0.8})
ax.set_title('Matrice de corrélation — variables numériques', fontsize=13, pad=12)
plt.tight_layout()
plt.show()

In [ ]:
# ── Boxplot des prix par ville ────────────────────────────────────────────────
city_order = (train.groupby('city')['log_price']
              .median().sort_values(ascending=False).index.tolist())

fig, ax = plt.subplots(figsize=(14, 6))
palette_cities = sns.color_palette('RdYlGn_r', n_colors=len(city_order))
sns.boxplot(data=train, x='city', y='log_price', order=city_order,
            palette=palette_cities, linewidth=1.2, fliersize=2, ax=ax)
ax.set_xticklabels(city_order, rotation=30, ha='right', fontsize=10)
ax.set_ylabel('log_price (= ln du prix en $)', fontsize=11)
ax.set_xlabel('')
ax.set_title('Distribution des prix par ville — du plus cher au moins cher', fontsize=13)
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# ── Amenities qui font monter / baisser le prix ───────────────────────────────
def _parse_am(s):
    if pd.isna(s): return []
    return [i.strip().strip('"') for i in str(s).strip('{}').split(',') if i.strip().strip('"')]

am_prices_dict = {}
for _, row in train.iterrows():
    for am in _parse_am(row['amenities']):
        if am not in am_prices_dict:
            am_prices_dict[am] = []
        am_prices_dict[am].append(row['log_price'])

am_df = pd.DataFrame([
    {'amenity': am, 'avg_log_price': np.mean(v), 'count': len(v)}
    for am, v in am_prices_dict.items() if len(v) >= 200
]).sort_values('avg_log_price', ascending=False)

global_mean_price = train['log_price'].mean()
top_am  = am_df.head(15).copy()
bot_am  = am_df.tail(10).copy()
plot_df = pd.concat([top_am, bot_am]).reset_index(drop=True)
plot_df['delta'] = plot_df['avg_log_price'] - global_mean_price
plot_df['color'] = plot_df['delta'].apply(lambda x: '#e74c3c' if x > 0 else '#2ecc71')

fig, ax = plt.subplots(figsize=(12, 9))
ax.barh(range(len(plot_df)), plot_df['delta'].values,
        color=plot_df['color'].values, edgecolor='white', linewidth=0.5)
ax.axvline(0, color='black', linewidth=1)
ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels(plot_df['amenity'].values, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel('Écart au prix moyen (en log_price)', fontsize=11)
ax.set_title('Amenities qui font monter (rouge) ou baisser (vert) le prix', fontsize=13)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Violin + courbe prix / voyageurs ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

room_order = (train.groupby('room_type')['log_price']
              .median().sort_values(ascending=False).index.tolist())
sns.violinplot(data=train, x='room_type', y='log_price', order=room_order,
               palette='coolwarm', inner='quartile', ax=axes[0])
axes[0].set_title('Distribution fine par type de logement', fontsize=12)
axes[0].set_xlabel('')
axes[0].set_ylabel('log_price')
axes[0].tick_params(axis='x', rotation=15)

acc_tmp = train.copy()
acc_tmp['accommodates'] = pd.to_numeric(acc_tmp['accommodates'], errors='coerce')
acc_tmp = acc_tmp.dropna(subset=['accommodates'])
acc_tmp['acc_bin'] = acc_tmp['accommodates'].clip(1, 10).astype(int)
acc_means = acc_tmp.groupby('acc_bin')['log_price'].mean()

axes[1].plot(acc_means.index, acc_means.values, 'o-', color='steelblue', linewidth=2, markersize=8)
axes[1].fill_between(acc_means.index, acc_means.values, alpha=0.15, color='steelblue')
axes[1].set_xlabel('Nombre de voyageurs', fontsize=11)
axes[1].set_ylabel('log_price moyen', fontsize=11)
axes[1].set_title('Prix moyen selon le nombre de voyageurs', fontsize=12)
axes[1].grid(alpha=0.4)
axes[1].set_xticks(acc_means.index)
plt.tight_layout()
plt.show()

### Ce qu'on retient de l'exploration

1. **La localisation** est clé : deux logements dans la même ville mais dans des quartiers différents peuvent avoir des prix très différents
2. **La taille** du logement est fortement liée au prix
3. **Les équipements** peuvent faire monter le prix (doorman, salle de sport, jacuzzi...)
4. **Le type** de logement (entier vs chambre) est très discriminant

Ces observations nous guident pour choisir quelles features créer.

## 3. Feature Engineering

Le modèle ne peut lire que des nombres. On transforme donc les colonnes texte, dates et listes en valeurs numériques.

| Colonne brute | Transformation | Pourquoi |
|---------------|----------------|----------|
| `amenities` (liste d'équipements) | Une colonne 0/1 par équipement | Permet au modèle de savoir si le logement a la WiFi, un parking, etc. |
| `host_since`, `first_review`... (dates) | Nombre de jours avant 2018 | Le modèle ne lit pas les dates, mais un nombre oui |
| `host_has_profile_pic` (`t`/`f`) | 0 ou 1 | Conversion simple |
| `description` (texte) | Longueur en caractères | Une description longue = hôte plus soigneux |
| `host_response_rate` (`90%`) | 0.9 | Conversion en nombre décimal |
| `zipcode`, `neighbourhood` | Nettoyage | Pour l'encodage qui suit |

In [ ]:
def parse_amenities(s):
    if pd.isna(s) or s == '':
        return []
    items = []
    for item in str(s).strip('{}').split(','):
        item = item.strip().strip('"').strip()
        if item:
            items.append(item)
    return items

# Top 30 amenities (calculé sur le train uniquement)
all_am = []
for a in train['amenities']:
    all_am.extend(parse_amenities(a))
am_counts = Counter(all_am)
TOP_AMENITIES = [a for a, _ in am_counts.most_common(30)]

print("Top 30 amenities :")
for i, (a, c) in enumerate(am_counts.most_common(30), 1):
    print(f"  {i:2d}. {a:<45} ({c:,})")

In [ ]:
def engineer_features(df):
    # Applique toutes les transformations de features sur train ET test
    df = df.copy()

    # ── Amenities ─────────────────────────────────────────────────────────
    # Colonne brute : {"TV","Wifi","Kitchen"} — on extrait chaque équipement
    # comme variable binaire (1 = présent) pour les 30 plus fréquents du train
    parsed_am = df['amenities'].apply(parse_amenities)
    df['amenities_count'] = parsed_am.apply(len)  # nb total d'équipements
    for am in TOP_AMENITIES:
        col = 'am_' + re.sub(r'[^a-z0-9]', '_', am.lower())[:30]
        df[col] = parsed_am.apply(lambda x: 1 if am in x else 0)

    # ── Dates → jours écoulés ──────────────────────────────────────────────
    # Référence : 01/01/2018 (fin de la période des données)
    # On ajoute _missing car l'absence de date (pas d'avis) est une info utile
    ref = pd.Timestamp('2018-01-01')
    for dc in ['host_since', 'first_review', 'last_review']:
        parsed = pd.to_datetime(df[dc], errors='coerce')
        df[dc + '_days']    = (ref - parsed).dt.days
        df[dc + '_missing'] = parsed.isna().astype(int)

    # ── Booléens t/f → 0/1 ────────────────────────────────────────────────
    # Le format CSV Airbnb stocke les booléens comme 't' ou 'f'
    for c in ['host_has_profile_pic', 'host_identity_verified', 'instant_bookable']:
        df[c] = (df[c].astype(str).str.lower() == 't').astype(int)
    df['cleaning_fee'] = (df['cleaning_fee'].astype(str).str.lower() == 'true').astype(int)

    # ── Taux de réponse '90%' → 0.9 ───────────────────────────────────────
    def parse_rate(x):
        try:
            return float(str(x).replace('%','').strip()) / 100.0
        except Exception:
            return np.nan
    df['host_response_rate'] = df['host_response_rate'].apply(
        lambda x: np.nan if pd.isna(x) or str(x).strip() == '' else parse_rate(x))

    # ── Longueur du texte ──────────────────────────────────────────────────
    # Proxy de qualité : une description longue et soignée = hôte professionnel
    df['description_len']   = df['description'].fillna('').apply(len)
    df['description_words'] = df['description'].fillna('').apply(lambda x: len(x.split()))
    df['name_len']           = df['name'].fillna('').apply(len)

    # ── Conversion numérique ───────────────────────────────────────────────
    # Certaines colonnes sont lues comme 'object' à cause de valeurs mixtes
    for c in ['bathrooms','bedrooms','beds','review_scores_rating','number_of_reviews','accommodates']:
        df[c] = pd.to_numeric(df[c], errors='coerce')

    # ── Indicateur : pas de reviews ────────────────────────────────────────
    # Les nouvelles annonces sans note ont un comportement de prix différent
    df['review_missing'] = df['review_scores_rating'].isna().astype(int)

    # ── Nettoyage pour encodage ────────────────────────────────────────────
    # Standardisation du zip (5 chiffres) et du quartier pour réduire le bruit
    df['zipcode_clean']       = df['zipcode'].fillna('Unknown').astype(str).str.strip().str[:5]
    df['neighbourhood_clean'] = df['neighbourhood'].fillna('Unknown').astype(str).str.strip()

    return df

train_fe = engineer_features(train)
test_fe  = engineer_features(test)
print(f'Colonnes après feature engineering : {train_fe.shape[1]}')

## 4. Encodage des variables catégorielles

Le modèle ne peut pas lire des mots comme `New York` ou `Entire home`. On doit les convertir en nombres.

**Pour les quartiers et codes postaux** (des centaines de valeurs différentes) :
On remplace chaque quartier par le **prix moyen** des logements de ce quartier. Ainsi Manhattan devient par exemple 4.8 (le log_price moyen de ce quartier). Pour ne pas biaiser les résultats, cette moyenne est calculée en excluant à chaque fois les lignes qu'on est en train de valider.

**Pour les autres catégories** (type de chambre, ville, type de propriété...) :
On assigne simplement un numéro à chaque valeur : `Entire home` devient 0, `Private room` devient 1, etc.

In [ ]:
def target_encode(train_df, test_df, cols, target='log_price', n_splits=5, smoothing=10):
    global_mean = train_df[target].mean()
    for col in cols:
        enc_tr = np.full(len(train_df), global_mean, dtype=float)
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
        for tr_idx, val_idx in kf.split(train_df):
            fold_tr  = train_df.iloc[tr_idx]
            fold_val = train_df.iloc[val_idx]
            stats  = fold_tr.groupby(col)[target].agg(['mean','count'])
            smooth = (stats['count']*stats['mean'] + smoothing*global_mean) / (stats['count']+smoothing)
            enc_tr[val_idx] = fold_val[col].map(smooth).fillna(global_mean).values
        stats_all  = train_df.groupby(col)[target].agg(['mean','count'])
        smooth_all = (stats_all['count']*stats_all['mean'] + smoothing*global_mean) / (stats_all['count']+smoothing)
        train_df[col+'_te'] = enc_tr
        test_df[col+'_te']  = test_df[col].map(smooth_all).fillna(global_mean).values
    return train_df, test_df

def label_encode(train_df, test_df, cols):
    for col in cols:
        le = LabelEncoder()
        combined = pd.concat([train_df[col].fillna('Unknown').astype(str),
                               test_df[col].fillna('Unknown').astype(str)], ignore_index=True)
        le.fit(combined)
        train_df[col+'_le'] = le.transform(train_df[col].fillna('Unknown').astype(str))
        test_df[col+'_le']  = le.transform(test_df[col].fillna('Unknown').astype(str))
    return train_df, test_df

TE_COLS = ['neighbourhood_clean', 'zipcode_clean']
LE_COLS = ['property_type', 'room_type', 'bed_type', 'cancellation_policy', 'city']

train_enc, test_enc = target_encode(train_fe, test_fe, TE_COLS)
train_enc, test_enc = label_encode(train_enc, test_enc, LE_COLS)
print("Encodage terminé")

In [ ]:
NUM_FEAT = [
    'accommodates','bathrooms','bedrooms','beds',
    'number_of_reviews','review_scores_rating','latitude','longitude',
    'amenities_count','description_len','description_words','name_len',
    'host_response_rate',
    'host_since_days','first_review_days','last_review_days',
    'host_since_missing','first_review_missing','last_review_missing',
    'review_missing',
    'host_has_profile_pic','host_identity_verified','instant_bookable','cleaning_fee',
]
TE_FEAT  = [c+'_te' for c in TE_COLS]
LE_FEAT  = [c+'_le' for c in LE_COLS]
AM_FEAT  = [c for c in train_enc.columns if c.startswith('am_')]
ALL_FEAT = NUM_FEAT + TE_FEAT + LE_FEAT + AM_FEAT

print(f"Total features : {len(ALL_FEAT)}")
print(f"  Numériques     : {len(NUM_FEAT)}")
print(f"  Target encoded : {len(TE_FEAT)}")
print(f"  Label encoded  : {len(LE_FEAT)}")
print(f"  Amenities      : {len(AM_FEAT)}")

In [ ]:
X      = train_enc[ALL_FEAT].copy()
y      = train_enc['log_price'].copy()
X_test = test_enc[ALL_FEAT].copy()

medians = X.median()
X       = X.fillna(medians)
X_test  = X_test.fillna(medians)

print(f"X      : {X.shape}")
print(f"y      : {y.shape}  mean={y.mean():.3f}  std={y.std():.3f}")
print(f"X_test : {X_test.shape}")
print(f"NaN dans X      : {X.isna().sum().sum()}")
print(f"NaN dans X_test : {X_test.isna().sum().sum()}")

## 5. Exploration des Modèles et Configurations

Avant de choisir le modèle final, on teste plusieurs approches pour justifier nos choix.

1. **Comparaison de modèles** : Ridge, Random Forest, Gradient Boosting, LightGBM — même jeu de features, pour voir quel algorithme marche le mieux
2. **Comparaison de configurations** : du jeu de features minimal au complet — pour voir l'apport de chaque groupe de variables

On utilise une validation croisée 3-fold (plus rapide) pour comparer.

In [ ]:
# ── Comparaison de modèles (3-fold CV) ───────────────────────────────────────
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

kf3 = KFold(n_splits=3, shuffle=True, random_state=SEED)

# Liste des modèles à tester
candidates = {
    'Ridge (linéaire)': Pipeline([
        ('scaler', StandardScaler()),
        ('model', Ridge(alpha=10))
    ]),
    'Random Forest': RandomForestRegressor(
        n_estimators=100, max_depth=12, random_state=SEED, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingRegressor(
        n_estimators=100, learning_rate=0.1, max_depth=5, random_state=SEED
    ),
}
if USE_LGBM:
    import lightgbm as lgb
    candidates['LightGBM'] = lgb.LGBMRegressor(
        n_estimators=300, learning_rate=0.05, num_leaves=63,
        random_state=SEED, verbose=-1, n_jobs=-1
    )

results_models = {}
print('Comparaison des modèles...')
for name, model in candidates.items():
    scores = []
    for tr_idx, val_idx in kf3.split(X):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        model.fit(X_tr, y_tr)
        scores.append(np.sqrt(mean_squared_error(y_val, model.predict(X_val))))
    results_models[name] = np.mean(scores)
    print(f'  {name:<25} RMSE = {np.mean(scores):.4f}')

best_model = min(results_models, key=results_models.get)
print(f'\nMeilleur modèle : {best_model} (RMSE={results_models[best_model]:.4f})')

In [ ]:
# ── Graphique comparaison des modèles ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
names  = list(results_models.keys())
scores = list(results_models.values())
best_score = min(scores)
colors = ['#e74c3c' if s == best_score else '#aec6e8' for s in scores]

bars = ax.bar(names, scores, color=colors, edgecolor='white', linewidth=0.5)
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{score:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('RMSE sur log_price — plus bas = meilleur')
ax.set_title('Comparaison des modèles (CV 3-fold, toutes les features)')
ax.set_ylim(min(scores) * 0.97, max(scores) * 1.02)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### Observations : comparaison des modèles

- Le modèle linéaire (Ridge) est clairement moins performant : la relation entre les features et le prix n'est pas linéaire
- Random Forest et Gradient Boosting donnent de bons résultats car ils capturent des interactions non-linéaires entre les variables
- LightGBM obtient le meilleur score avec les mêmes features — c'est donc lui qu'on utilisera pour le modèle final

On garde LightGBM pour la suite.

In [ ]:
# ── Comparaison de configurations de features ────────────────────────────────
# On part du plus simple et on ajoute des groupes de variables progressivement

FEAT_BASE  = ['accommodates','bathrooms','bedrooms','beds',
              'number_of_reviews','review_scores_rating','latitude','longitude']
FEAT_CAT   = FEAT_BASE + LE_FEAT + TE_FEAT
FEAT_AM    = FEAT_CAT + AM_FEAT
FEAT_FULL  = ALL_FEAT  # tout

configs = {
    'Taille + localisation (8 features)': FEAT_BASE,
    '+ Catégories encodées':               FEAT_CAT,
    '+ Amenities (équipements)':            FEAT_AM,
    'Toutes les features':                  FEAT_FULL,
}

if USE_LGBM:
    base_model = lgb.LGBMRegressor(
        n_estimators=300, learning_rate=0.05, num_leaves=63,
        random_state=SEED, verbose=-1, n_jobs=-1
    )
else:
    base_model = GradientBoostingRegressor(
        n_estimators=100, learning_rate=0.1, max_depth=5, random_state=SEED
    )

results_configs = {}
print('Comparaison des configurations...')
for name, feats in configs.items():
    X_cfg = X[feats].fillna(X[feats].median())
    scores = []
    for tr_idx, val_idx in kf3.split(X_cfg):
        X_tr, X_val = X_cfg.iloc[tr_idx], X_cfg.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        import copy
        m = copy.deepcopy(base_model)
        m.fit(X_tr, y_tr)
        scores.append(np.sqrt(mean_squared_error(y_val, m.predict(X_val))))
    results_configs[name] = (np.mean(scores), len(feats))
    print(f'  {name:<40} {len(feats):3d} features  RMSE={np.mean(scores):.4f}')

In [ ]:
# ── Graphique : impact des groupes de features ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cfg_names  = list(results_configs.keys())
cfg_rmse   = [v[0] for v in results_configs.values()]
cfg_nfeat  = [v[1] for v in results_configs.values()]
best_rmse  = min(cfg_rmse)
colors = ['#e74c3c' if s == best_rmse else '#aec6e8' for s in cfg_rmse]

# Barres RMSE
bars = axes[0].bar(range(len(cfg_names)), cfg_rmse, color=colors,
                   edgecolor='white', linewidth=0.5)
for bar, score in zip(bars, cfg_rmse):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                 f'{score:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[0].set_xticks(range(len(cfg_names)))
axes[0].set_xticklabels(cfg_names, rotation=20, ha='right', fontsize=9)
axes[0].set_ylabel('RMSE — plus bas = meilleur')
axes[0].set_title('RMSE selon la configuration de features')
axes[0].set_ylim(min(cfg_rmse)*0.97, max(cfg_rmse)*1.02)
axes[0].grid(axis='y', alpha=0.3)

# Amélioration progressive
axes[1].plot(range(len(cfg_names)), cfg_rmse, 'o-', color='steelblue',
             linewidth=2, markersize=8)
axes[1].fill_between(range(len(cfg_names)), cfg_rmse, alpha=0.15, color='steelblue')
axes[1].set_xticks(range(len(cfg_names)))
axes[1].set_xticklabels(cfg_names, rotation=20, ha='right', fontsize=9)
axes[1].set_ylabel('RMSE')
axes[1].set_title('Amélioration progressive en ajoutant des features')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Tableau récapitulatif de tous les résultats ───────────────────────────────
rows = []
for name, rmse in results_models.items():
    rows.append({'Expérience': name, 'Type': 'Modèle', 'RMSE (CV 3-fold)': round(rmse, 4)})
for name, (rmse, nf) in results_configs.items():
    rows.append({'Expérience': f'{name} ({nf} feat.)', 'Type': 'Config features',
                 'RMSE (CV 3-fold)': round(rmse, 4)})

summary = pd.DataFrame(rows).sort_values('RMSE (CV 3-fold)')
print('=== Récapitulatif des expériences ===')
print(summary.to_string(index=False))

### Conclusions des expériences

- **LightGBM** donne le meilleur score parmi tous les modèles testés
- Chaque groupe de features supplémentaire améliore le score : les catégories encodées (quartier, ville) apportent le plus gros gain, suivi des amenities et des features texte/hôte
- Utiliser **toutes les features + LightGBM** est donc la meilleure combinaison

On passe maintenant à l'entraînement final avec cette configuration.

## 6. Modèle Final

### Pourquoi LightGBM ?
C'est un algorithme qui construit plein de petits arbres de décision à la suite, chacun corrigeant les erreurs du précédent. On l'a choisi car il donne de très bons résultats sur des données tabulaires comme les nôtres, et il gère bien les valeurs manquantes.

### Validation croisée (5-fold)
Pour évaluer le modèle sans toucher aux données de test, on divise le train en 5 parties égales. On entraîne 5 fois : à chaque fois, 4 parties servent à apprendre et la 5ème sert à mesurer la performance. Cela donne une note fiable sur des données que le modèle n'a pas vues.

L'entraînement s'arrête automatiquement quand les performances ne s'améliorent plus — pour éviter de surapprendre.

### Paramètres principaux
| Paramètre | Valeur | Effet |
|-----------|--------|-------|
| `learning_rate` | 0.05 | Apprentissage lent mais précis |
| `num_leaves` | 63 | Arbres ni trop simples ni trop complexes |
| `feature_fraction` | 0.8 | Utilise 80% des features à chaque arbre |
| `bagging_fraction` | 0.8 | Utilise 80% des données à chaque arbre |

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
oof_preds = np.zeros(len(X))
cv_scores = []

if USE_LGBM:
    params = dict(
        objective='regression', metric='rmse',
        learning_rate=0.05, num_leaves=63, max_depth=-1,
        min_child_samples=20, feature_fraction=0.8,
        bagging_fraction=0.8, bagging_freq=5,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=SEED, verbose=-1, n_jobs=-1,
    )
    best_iters = []

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        dtrain = lgb.Dataset(X_tr, label=y_tr, feature_name=ALL_FEAT)
        dval   = lgb.Dataset(X_val, label=y_val)
        model  = lgb.train(params, dtrain, num_boost_round=3000,
                           valid_sets=[dval],
                           callbacks=[lgb.early_stopping(150, verbose=False),
                                      lgb.log_evaluation(500)])
        preds = model.predict(X_val)
        oof_preds[val_idx] = preds
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse); best_iters.append(model.best_iteration)
        print(f"  Fold {fold+1}/5 | RMSE={rmse:.4f} | best_iter={model.best_iteration}")

    best_n = int(np.mean(best_iters) * 1.05)
    print(f"\nMoyenne best_iter={int(np.mean(best_iters))} → modèle final : {best_n} rounds")

else:
    from sklearn.ensemble import GradientBoostingRegressor
    best_n = 300
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        m = GradientBoostingRegressor(n_estimators=300, learning_rate=0.1,
                                       max_depth=5, subsample=0.8, random_state=SEED)
        m.fit(X_tr, y_tr); preds = m.predict(X_val)
        oof_preds[val_idx] = preds
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        print(f"  Fold {fold+1}/5 | RMSE={rmse:.4f}")

print(f"\n{'='*40}")
print(f"CV RMSE : {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")
print(f"{'='*40}")

### Résultats de la validation croisée

Un RMSE d'environ 0.44 sur le log_price correspond à une erreur d'environ 55% sur le prix réel — ce qui est attendu pour ce type de données très variables.

Les scores sont similaires sur les 5 folds, donc le modèle est stable. Le modèle final est ensuite entraîné sur toutes les données disponibles.

In [ ]:
# Modèle final entraîné sur tout le train
print(f"Entraînement final ({best_n} itérations)...")

if USE_LGBM:
    dtrain_full = lgb.Dataset(X, label=y, feature_name=ALL_FEAT)
    final_model = lgb.train(params, dtrain_full, num_boost_round=best_n)
    test_preds  = final_model.predict(X_test)
else:
    final_model = GradientBoostingRegressor(n_estimators=best_n, learning_rate=0.1,
                                             max_depth=5, subsample=0.8, random_state=SEED)
    final_model.fit(X, y)
    test_preds = final_model.predict(X_test)

print(f"Prédictions test — min={test_preds.min():.3f}  max={test_preds.max():.3f}  "
      f"mean={test_preds.mean():.3f}")

## 7. Analyse du Modèle

On vérifie que le modèle a bien appris en regardant :
1. **Quelles variables ont été les plus utiles ?**
2. **Les prédictions sont-elles proches des vraies valeurs ?**
3. **Les erreurs sont-elles aléatoires** ou y a-t-il un pattern ?

In [ ]:
# Feature importance
if USE_LGBM:
    imp = pd.DataFrame({'feature': ALL_FEAT,
                        'importance': final_model.feature_importance(importance_type='gain')})
else:
    imp = pd.DataFrame({'feature': ALL_FEAT,
                        'importance': final_model.feature_importances_})

imp = imp.sort_values('importance', ascending=False).reset_index(drop=True)
top20 = imp.head(20)

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(top20)))
ax.barh(range(len(top20)), top20['importance'].values, color=colors)
ax.set_yticks(range(len(top20))); ax.set_yticklabels(top20['feature'].values, fontsize=10)
ax.invert_yaxis(); ax.set_title('Top 20 features (importance gain)', fontsize=13)
plt.tight_layout(); plt.show()

print("Top 10 features:")
print(imp.head(10).to_string(index=False))

### Ce que nous dit l'importance des features

- **La localisation** (quartier, code postal, latitude/longitude) est de loin la variable la plus importante — cohérent avec l'EDA
- **La taille** du logement (capacité, chambres, salles de bain) arrive en deuxième position
- **L'ancienneté** de l'annonce (date du premier avis, date d'inscription) a aussi un impact
- Certains **équipements** apparaissent dans le top 20, ce qui confirme que les extraire était pertinent

In [ ]:
# OOF : prédictions vs réel + distribution des résidus
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y, oof_preds, alpha=0.1, s=1, color='steelblue')
axes[0].plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2, label='Parfait')
axes[0].set_xlabel('log_price réel'); axes[0].set_ylabel('log_price prédit (OOF)')
axes[0].set_title('OOF : prédit vs réel'); axes[0].legend()

residuals = y - oof_preds
axes[1].hist(residuals, bins=60, color='coral', edgecolor='white', lw=0.5)
axes[1].axvline(0, color='black', ls='--')
axes[1].set_title(f'Résidus (std={residuals.std():.3f})')
axes[1].set_xlabel('Résidu (réel − prédit)')

plt.tight_layout(); plt.show()
oof_rmse = np.sqrt(mean_squared_error(y, oof_preds))
print(f"OOF RMSE global : {oof_rmse:.4f}")

### Interprétation des résidus

- Les points s'alignent bien sur la diagonale : les prédictions sont globalement proches des vraies valeurs
- Les erreurs sont réparties de façon symétrique autour de 0 : le modèle ne surestime pas systématiquement
- Le modèle est un peu moins précis sur les prix très élevés ou très bas, ces cas étant moins représentés dans les données

## 8. Export des prédictions

In [ ]:
submission = pd.DataFrame({'logpred': test_preds}, index=test_enc['id'].values)
submission.index.name = ''
submission.to_csv('predictions.csv')
print(f"Sauvegardé : predictions.csv  ({len(submission):,} lignes)")
submission.head()

In [ ]:
# Vérification du format
example = pd.read_csv('prediction_example.csv', index_col=0)
pred    = pd.read_csv('predictions.csv', index_col=0)

print("=== Vérification ===")
print(f"Colonnes attendues : {list(example.columns)}")
print(f"Colonnes obtenues  : {list(pred.columns)}")
print(f"Lignes             : {len(pred):,}")
print()
print(pred.head(5))
print()

plt.figure(figsize=(10, 4))
plt.hist(pred['logpred'], bins=60, color='steelblue', edgecolor='white', lw=0.5, label='Test (prédit)')
plt.axvline(y.mean(), color='red', ls='--', label=f'Moyenne train ({y.mean():.2f})')
plt.title('Distribution des prédictions'); plt.xlabel('log_price prédit')
plt.legend(); plt.tight_layout(); plt.show()
print("Export OK!")

## 9. Conclusion

### Ce qu'on a fait
```
Exploration -> Préparation des features -> Encodage -> LightGBM -> Export
```

### Résultats
- RMSE d'environ 0.44 sur le log_price, stable sur les 5 folds
- Les variables les plus utiles : la localisation (quartier, coordonnées GPS) et la taille du logement

### Ce qui a bien marché
- Remplacer les quartiers par leur prix moyen a beaucoup aidé le modèle
- Extraire les équipements comme variables 0/1 a enrichi les données
- La validation croisée donne une estimation fiable de la performance

### Ce qu'on pourrait améliorer
- Analyser le texte de la description pour en extraire plus d'information
- Combiner plusieurs modèles pour améliorer les prédictions
- Ajouter des données sur les quartiers (transports, restaurants...)